# Corporacion Favorita - New Superb Forecasting Model - 

## Split and Model Pipeline

#codi

Made by 4B Consultancy (Janne Heuvelmans, Georgi Duev, Alexander Engelage, Sebastiaan de Bruin) - 2024

In this data pipeline, 

The following steps are made within this notebook:  

>-0. Import Packages 

>-1. Load final dataset and aggregate dataset to weekly level
    -1.1 Load final dataset made in Data Preperation Pipeline Notebook
    -1.2 Aggregate dataset to weekly level

>-2. Column transformers and Train, Test, Validation Split

>-3. Models

>-4. Pick best model one and optimize with grid search

## 0. Import Packages

In [1]:
# Importing the libraries
import pandas as pd
import numpy as np
import polars as pl
import os
import sys
import altair as alt
import vegafusion as vf
import sklearn
import time
from datetime import date, datetime, timedelta
from sklearn.pipeline import Pipeline, make_pipeline

In [2]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

from sklearn.metrics import mean_absolute_percentage_error

import statsmodels.api as sm

In [3]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

In [4]:
from sktime.forecasting.compose import EnsembleForecaster

## 1. Load final dataset

### 1.1. Functions - Import raw data from local PATH
Create import data function and give basic information function within the importing function.

Return basic information on each dataframe:  
- a) Information on the number of observation and features.  
- b) Information on the size of the dataframe. 

TO-DO: Import via polars, and use polars dataframe?

In [5]:
def f_get_data_and_info(import_path, file_name):

    print(f"\nReading file {file_name}\n")

    # Load data.
    df = pd.read_parquet(import_path + file_name + ".parquet")

    # Getting the basic information of the dataframe (number of observations and features, and size)
    print(
        f"The '{file_name}' dataframe contains: {df.shape[0]:,}".replace(",", ".")
        + f" observations and {df.shape[1]} features."
    )
    print(
        f"Prepared and transformed dataframe has optimized size of {round(sys.getsizeof(df)/1024/1024/1024, 2)} GB."
    )

    return df

### 1.2. Importing raw data
Importing parquet files with importing function (giving basic information)

In [6]:
import_path = "C:/Users/alexander/Documents/0. Data Science and AI for Experts/TEST/"

# Importing final df
df_final = f_get_data_and_info(import_path, file_name="Prepped_data_20241004")


Reading file Prepped_data_20241004

The 'Prepped_data_20241004' dataframe contains: 67.834.270 observations and 19 features.
Prepared and transformed dataframe has optimized size of 2.72 GB.


To-do: include null_count print dunction into importing OR make basic descrption function with features, size, null_count

In [7]:
df_final.info()
# Count nulls per column
null_counts = df_final.isnull().sum()

# Print results
for column, count in null_counts.items():
    print(f"Column '{column}' has {count} null values.")

<class 'pandas.core.frame.DataFrame'>
Index: 67834270 entries, 0 to 67834269
Data columns (total 19 columns):
 #   Column                  Dtype         
---  ------                  -----         
 0   store_nbr               uint8         
 1   item_nbr                int32         
 2   date                    datetime64[ns]
 3   unit_sales              float32       
 4   onpromotion             bool          
 5   holiday_local_count     int8          
 6   holiday_national_count  int8          
 7   holiday_regional_count  int8          
 8   store_type              category      
 9   store_cluster           uint8         
 10  item_family             category      
 11  item_class              uint16        
 12  perishable              uint8         
 13  store_status            int8          
 14  item_status             int8          
 15  year                    int16         
 16  weekday                 int8          
 17  week_nbr                int8          
 18  week_

## 2.0 Train val test split

SKtime

ExpandingWindowSplitter


#TO-DO: Selecting on weeks or via data?

train_val_test_split without creating X (features) and y (target)

In [8]:
def train_val_test_split(df, window_length=26):

    # Sort the DataFrame by store number, item number, and date for ordering
    df = df.sort_values(["store_nbr", "item_nbr", "week_number_cum"])

    # Get the maximum week in the dataset
    max_week = df["week_number_cum"].max()

    # Calculate start and end weeks for validation and test sets
    test_week_start = max_week - window_length + 1

    val_week_start = max_week - 2 * window_length + 1

    val_week_end = test_week_start - 1

    train_week_end = val_week_start - 1

    # Train data: All data before the start of the validation period
    train = df[df["week_number_cum"] <= train_week_end]

    # Val data: From `val_week_start` to `val_week_end`
    val = df[
        (df["week_number_cum"] >= val_week_start)
        & (df["week_number_cum"] <= val_week_end)
    ]

    # Test data: From `test_week_start` to `max_week`
    test = df[
        (df["week_number_cum"] >= test_week_start) & (df["week_number_cum"] <= max_week)
    ]

    # Function to print split information
    def print_split_info(split_name, split):
        print(f"\n{split_name} set: shape: {split.shape}")
        print(f"{split_name} Min Week: {split['week_number_cum'].min()}")
        print(f"{split_name} Max Week: {split['week_number_cum'].max()}")
        print(f"{split_name} number of weeks: {split['week_number_cum'].nunique()}")
        print(f"Number of stores: {split['store_nbr'].nunique()}")
        print(f"Number of items: {split['item_nbr'].nunique()}")

    # Print information about the splits
    print_split_info("Train", train)
    print_split_info("Validation", val)
    print_split_info("Test", test)

    return train, val, test

In [9]:
# train, val, test = train_val_test_split(df_final, window_length=26)

## 3.0 Functions - Impute stockouts and Aggregate dataset to weekly level


#### 3.1. Impute stockouts

Stockout on store level

•      Perishable good: when there are missing values for two consecutive days for a given item per individual store 

•      Nonperishable goods: when there are missing values for 7 consecutive days for a given item and per individual store

•      Action: Impute with Rolling Mean with defeault window of 7 days 

------------------------------------

In [10]:
def impute_stockouts_polars(df_pandas, window_size=7):

    # Convert the input Pandas DataFrame to a Polars DataFrame
    df = pl.from_pandas(df_pandas)

    # Sort the DataFrame by store number, item number, and date for ordering

    df = df.sort(["store_nbr", "item_nbr", "date"])

    # Nested function calc_missing_count to calculate the count of consecutive missing values in unit_sales

    def calc_missing_count(unit_sales):

        return (
            unit_sales.is_null()  # Check for null values
            .cast(pl.Int32)  # Cast to integer (1 for null, 0 for not null)
            .cum_sum()  # Cumulative sum to count sequential nulls
            .over(["store_nbr", "item_nbr"])  # Group by store_nbr and item_nbr
        )

    # Nested function to Inpute with rolling mean for missing values
    def rolling_mean_imputation(unit_sales, window_size):

        return (
            unit_sales.rolling_mean(
                window_size=window_size, min_periods=1
            )  # Impute strategy based on rolling mean
            .shift(
                1
            )  # Shift window by one day, to prevent taking the same day into account
            .over(["store_nbr", "item_nbr"])  # Group by store_nbr and item_nbr
        )

    # Apply the imputation logic based on the perishable status of the items

    df = df.with_columns(
        [
            pl.when(pl.col("perishable") == 1)  # Check if the item is perishable = 1
            .then(
                pl.when(
                    calc_missing_count(pl.col("unit_sales")) == 1
                )  # 1 missing value
                .then(0)  # --> Impute with 0
                .when(
                    calc_missing_count(pl.col("unit_sales")) > 2
                )  # More than 2 missing values
                .then(0)  # --> Impute with 0
                .when(
                    calc_missing_count(pl.col("unit_sales")) == 2
                )  # = 2 missing values
                .then(
                    rolling_mean_imputation(pl.col("unit_sales"), window_size)
                )  # --> Inpute with rolling mean for 2 missing days
                .otherwise(pl.col("unit_sales"))  # Otherwise keep original value
            )
            .when(pl.col("perishable") == 0)  # If the item is not perishable = 0
            .then(
                pl.when(
                    calc_missing_count(pl.col("unit_sales")) > 7
                )  # More than 7 missing values
                .then(0)  # --> Impute with 0
                .when(
                    calc_missing_count(pl.col("unit_sales")) <= 7
                )  # if less 7 missing values
                .then(
                    rolling_mean_imputation(pl.col("unit_sales"), window_size)
                )  # --> Inpute with rolling mean for missing 7 or less days
                .otherwise(pl.col("unit_sales"))  # Otherwise keep original value
            )
            .otherwise(pl.col("unit_sales"))  # For any other case not covered
            .alias("unit_sales")  # Alias the new column as 'unit_sales'
        ]
    )

    # Convert Polars df back to Pandas df
    df = df.to_pandas()

    return df

### 3.2. Aggregate dataset to weekly level

- Group the DataFrame by store number, item number, year, and week_cum_number, then aggregate the columns
--> "unit_sales","onpromotion", "holiday_local_count","holiday_regional_count","holiday_national_count",


In [11]:
def aggregate_week(df):

    # Sort the DataFrame by store number, item number, and date for ordering
    df = df.sort_values(["store_nbr", "item_nbr", "year", "week_nbr"])

    # Group by the specified columns and aggregate
    df = (
        df.groupby(
            [
                "store_nbr",
                "item_nbr",
                "year",
                "week_number_cum",  # Aggregating by week_number_cum
            ]
        )
        .agg(
            {
                "unit_sales": "sum",
                "onpromotion": "sum",
                "holiday_local_count": "sum",
                "holiday_regional_count": "sum",
                "holiday_national_count": "sum",
                "date": "first",  # Keep the first day of week, needed to run Timeseries models from SKtime
                "store_type": "first",  # Keep the first occurrence of store_type
                "store_cluster": "first",  # Keep the first occurrence of store_cluster
                "item_family": "first",  # Keep the first occurrence of item_family
                "item_class": "first",  # Keep the first occurrence of item_class
                "perishable": "first",  # Keep the first occurrence of perishable
                "store_status": "last",  # Keep the last occurrence of store_status
                "item_status": "last",  # Keep the last occurrence of item_status
            }
        )
        .reset_index()
    )

    return df

## 4. Pipeline and preprocessing

Splitting and preprocessing with imputation and aggregating to weekly data

In [12]:
features = ["date", "store_nbr", "item_nbr"]

target_variable = ["unit_sales"]

In [13]:
def impute_agg_preprocessing(df, window_size=7):

    df = impute_stockouts_polars(df, window_size)

    df = aggregate_week(df_final)

    return df

In [ ]:
def preprocess_split_filer(df, features, target_variable):

    # Splitting in train, validation, test split
    train_df, val_df, test_df = train_val_test_split(df)

    # Preprocessing with imputation and aggregating to weekly data
    train_df = impute_agg_preprocessing(train_df)
    val_df = impute_agg_preprocessing(val_df)
    test_df = impute_agg_preprocessing(test_df)

    # Filter spilt_df on needed feature and target variables
    train_df = train_df[features + target_variable]
    val_df = val_df[features + target_variable]
    test_df = test_df[features + target_variable]

    # Ensure df's are sorted by store, item, and date for accurate alignment
    train_df = train_df.sort_values(by=["store_nbr", "item_nbr", "date"])
    val_df = val_df.sort_values(by=["store_nbr", "item_nbr", "date"])
    test_df = test_df.sort_values(by=["store_nbr", "item_nbr", "date"])

    return train_df, val_df, test_df

In [15]:
train_df, val_df, test_df = preprocess_split_filer(df_final, features, target_variable)


Train set: shape: (53398880, 19)
Train Min Week: 1
Train Max Week: 190
Train number of weeks: 190
Number of stores: 10
Number of items: 4021

Validation set: shape: (7318220, 19)
Validation Min Week: 191
Validation Max Week: 216
Validation number of weeks: 26
Number of stores: 10
Number of items: 4021

Test set: shape: (7117170, 19)
Test Min Week: 217
Test Max Week: 242
Test number of weeks: 26
Number of stores: 10
Number of items: 4021


In [16]:
def null_count(df):  # Count nulls per column
    null_counts = df.isnull().sum()

    # Print results
    for column, count in null_counts.items():
        print(f"Column '{column}' has {count} null values.")

    return

In [17]:
null_count(train_df)

Column 'date' has 0 null values.
Column 'store_nbr' has 0 null values.
Column 'item_nbr' has 0 null values.
Column 'unit_sales' has 0 null values.


## 5. Model Pipeline

### 5.1 Model: Holt-Winters

In [ ]:
# holt_winters_model = ExponentialSmoothing(
#     df_train, trend="add", seasonal="add", seasonal_periods=12
# ).fit()

#                   .fit(smoothing_level=0.5, #=best_alpha
#                        smoothing_slope=0.5, #=best_beta
#                        smoothing_seasonal=0.5) #=best_gamma

NameError: name 'df_train' is not defined

In [ ]:
DEF STOP

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)  # Suppress FutureWarnings

### 5.2 Run Model with Polars

In [29]:
def holt_winters_model_forecast(
    train_df, val_df, seasonal_periods=12, trend="add", seasonal="add"
):

    # Convert the input Pandas DataFrame to a Polars DataFrame
    train_df = pl.from_pandas(train_df)
    val_df = pl.from_pandas(test_df)

    unique_stores = train_df["store_nbr"].unique()
    unique_items = train_df["item_nbr"].unique()

    forecasts = {}

    for store in unique_stores:
        print(
            f"Model Holly starts training to Winnie the forecasting for store {store}"
        )
        for item in unique_items:
            # Filter the data for the specific store and item using polars filtering
            train_data = train_df.filter(
                (pl.col("store_nbr") == store) & (pl.col("item_nbr") == item)
            )
            train_data = train_data.sort("date")

            # Convert Polars DataFrame to Pandas (needed for statsmodels, as it only runs in pandas)
            # Set index on date for unit_sales, per unique store, per item
            train_series = train_data.to_pandas().set_index("date")["unit_sales"]

            # Fit Holt Winters Model
            try:
                model = ExponentialSmoothing(
                    train_series,
                    trend=trend,
                    seasonal=seasonal,
                    seasonal_periods=seasonal_periods,
                )
                fitted_model = model.fit()

                # Forecast the validation period length
                forecast_length = len(
                    val_df.filter(pl.col("store_nbr") == store)["date"].unique()
                )
                forecast = fitted_model.forecast(forecast_length)

                # Store the forecast in a dictionary for future reference
                forecasts[(store, item)] = forecast

            except Exception as e:
                print(f"Model failed for store {store} item {item}: {e}")

    return forecasts

In [ ]:
forecast_predictions = holt_winters_model_forecast(
    train_df, val_df, seasonal_periods=12, trend="add", seasonal="add"
)  # --> Takes 85 minutes for 10 stores

Model starts training and forecasting for store 1
Model starts training and forecasting for store 2
Model starts training and forecasting for store 3
Model starts training and forecasting for store 4
Model starts training and forecasting for store 5
Model starts training and forecasting for store 6
Model starts training and forecasting for store 7
Model starts training and forecasting for store 8
Model starts training and forecasting for store 9
Model starts training and forecasting for store 10


### 5.3. Evaulation Metrics and Evaluate Model functions

In [30]:
def calculate_metrics(y_true, y_pred):

    if len(y_true) == 0 or len(y_pred) == 0:
        return {"MAPE": np.nan, "Accuracy": np.nan, "Bias": np.nan}

    mape = mean_absolute_percentage_error(y_true, y_pred)
    accuracy = 1 - mape
    bias = np.mean(y_pred - y_true)
    return {"MAPE": mape, "Accuracy": accuracy, "Bias": bias}

In [ ]:
def evaluate_forecasts(forecasts, val_df):

    # Initialize lists to store metrics for each store and item
    all_metrics = []
    store_metrics = []

    # Ensure val_df is sorted by store, item, and date for accurate alignment
    val_df = val_df.sort_values(by=["store_nbr", "item_nbr", "date"])

    # Iterate over each store and item combination to collect true and predicted values
    for (store, item), forecast in forecasts.items():

        y_true = val_df[(val_df["store_nbr"] == store) & (val_df["item_nbr"] == item)][
            "unit_sales"
        ]

        # Index the values
        y_true = y_true.reset_index(drop=True)
        y_pred = pd.Series(forecast).reset_index(drop=True)

        if len(y_true) == len(y_pred):
            metrics = calculate_metrics(y_true, y_pred)
            metrics.update({"store_nbr": store, "item_nbr": item})
            all_metrics.append(metrics)

            if store not in store_metrics:
                store_metrics[store] = {"MAPE": [], "Accuracy": [], "Bias": []}
            store_metrics[store]["MAPE"].append(metrics["MAPE"])
            store_metrics[store]["Accuracy"].append(metrics["Accuracy"])
            store_metrics[store]["Bias"].append(metrics["Bias"])

    # Calculate average metrics for each store
    average_store_metrics = []
    for store, metrics in store_metrics.items():
        average_mape = np.nanmean(metrics["MAPE"])
        average_accuracy = np.nanmean(metrics["Accuracy"])
        average_bias = np.nanmean(metrics["Bias"])

        average_store_metrics.append(
            {
                "store_nbr": store,
                "item_nbr": "average",
                "MAPE": average_mape,
                "Accuracy": average_accuracy,
                "Bias": average_bias,
            }
        )

    # Calculate overall average metrics
    overall_mape = np.nanmean(
        [metric["MAPE"] for metric in all_metrics if not np.isnan(metric["MAPE"])]
    )
    overall_accuracy = np.nanmean(
        [
            metric["Accuracy"]
            for metric in all_metrics
            if not np.isnan(metric["Accuracy"])
        ]
    )
    overall_bias = np.nanmean(
        [metric["Bias"] for metric in all_metrics if not np.isnan(metric["Bias"])]
    )

    overall_metrics = {
        "store_nbr": "overall",
        "item_nbr": "overall",
        "MAPE": overall_mape,
        "Accuracy": overall_accuracy,
        "Bias": overall_bias,
    }

    # Convert metrics to df's
    metrics_df = pd.DataFrame(all_metrics)
    average_store_metrics_df = pd.DataFrame(average_store_metrics)
    overall_metrics_df = pd.DataFrame([overall_metrics])

    # Print metrics
    # print(metrics_df)
    # print(average_store_metrics_df)
    print(overall_metrics_df)

    return metrics_df, average_store_metrics_df, overall_metrics_df

In [45]:
# Evaluate forecasts and print metrics
metrics_df, average_store_metrics_df, overall_metrics_df = evaluate_forecasts(
    forecast_predictions, val_df
)

IndexError: list assignment index out of range

In [ ]:
overall_metrics_df

,store_nbr,item_nbr,MAPE,Accuracy,Bias
0,overall,overall,6.178246e+16,-6.178246e+16,-0.692417


In [33]:
average_store_metrics_df

,store_nbr,item_nbr,MAPE,Accuracy,Bias
0,1,average,2.865934e+16,-2.865934e+16,-0.692417
1,2,average,4.382604e+16,-4.382604e+16,NaN
2,3,average,1.805969e+17,-1.805969e+17,NaN
3,4,average,3.867717e+16,-3.867717e+16,NaN
4,5,average,3.832892e+16,-3.832892e+16,NaN
5,6,average,4.138107e+16,-4.138107e+16,NaN
6,7,average,6.461933e+16,-6.461933e+16,NaN
7,8,average,8.448565e+16,-8.448565e+16,NaN
8,9,average,7.365473e+16,-7.365473e+16,NaN
9,10,average,2.359547e+16,-2.359547e+16,NaN


In [ ]:
metrics_df.sort_values(by="MAPE", ascending=True).head(50)

,MAPE,Accuracy,Bias,store_nbr,item_nbr
8145,0.126926,0.873074,NaN,3,165704
8235,0.131111,0.868889,NaN,3,227111
21233,0.132312,0.867688,NaN,6,852110
8126,0.137070,0.862930,NaN,3,158680
8558,0.140540,0.859460,NaN,3,457425
4870,0.145241,0.854759,NaN,2,679926
24575,0.146788,0.853212,NaN,7,407499
20983,0.147097,0.852903,NaN,6,699703
24769,0.147324,0.852676,NaN,7,557256
574,0.148766,0.851234,NaN,1,502331


-------------------------------------------

### X.4 Hyperparameter Optimization (Grid Search approach)

https://www.kaggle.com/code/mehmetisik/smoothing-methods-holt-winters/notebook#Final-TES-Model

- alpha: smoothing level for the level components
- beta: smoothing level for the trend components
- gamma: smoothing level for the seasonal components


In [ ]:
# Create a range of values for alpha, beta, and gamma
alphas = betas = gammas = np.arange(0.20, 1, 0.10)

In [ ]:
import itertools

# create all combinations of alpha, beta, and gamma
abg = list(itertools.product(alphas, betas, gammas))

In [ ]:
def hw_model_optimizer(df_train, abg, step=24):

    best_alpha, best_beta, best_gamma, best_mape = None, None, None, float("inf")

    for comb in abg:

        model = ExponentialSmoothing(
            df_train, trend="add", seasonal="add", seasonal_periods=12
        ).fit(
            smoothing_level=comb[0], smoothing_slope=comb[1], smoothing_seasonal=comb[2]
        )

        y_pred = model.forecast(step)
        mape = mean_absolute_percentage_error(df_train[-step:], y_pred)
        if mape < best_mape:
            best_alpha, best_beta, best_gamma, best_mape = (
                comb[0],
                comb[1],
                comb[2],
                mape,
            )
        print([round(comb[0], 2), round(comb[1], 2), round(comb[2], 2), round(mape, 2)])

    print(
        "best_alpha:",
        round(best_alpha, 2),
        "best_beta:",
        round(best_beta, 2),
        "best_gamma:",
        round(best_gamma, 2),
        "best_mae:",
        round(best_mape, 4),
    )

    return best_alpha, best_beta, best_gamma, best_mape

In [ ]:
best_alpha, best_beta, best_gamma, best_mae = hw_model_optimizer(df_train, abg)

In [ ]:
final_tes_model = ExponentialSmoothing(
    train, trend="add", seasonal="add", seasonal_periods=12
).fit(
    smoothing_level=best_alpha, smoothing_trend=best_beta, smoothing_seasonal=best_gamma
)

Parallel Processing for item forecasts in parallel for each unique store

In [ ]:
import polars as pl
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import numpy as np
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error
import pandas as pd
import itertools
from joblib import Parallel, delayed


def hw_model_optimizer(df_train, abg, step=48):
    best_alpha, best_beta, best_gamma, best_mape = None, None, None, float("inf")
    for comb in abg:
        model = ExponentialSmoothing(
            df_train, trend="add", seasonal="add", seasonal_periods=12
        ).fit(
            smoothing_level=comb[0], smoothing_slope=comb[1], smoothing_seasonal=comb[2]
        )
        y_pred = model.forecast(step)
        mape = mean_absolute_percentage_error(df_train[-step:], y_pred)
        if mape < best_mape:
            best_alpha, best_beta, best_gamma, best_mape = (
                comb[0],
                comb[1],
                comb[2],
                mape,
            )
        print([round(comb[0], 2), round(comb[1], 2), round(comb[2], 2), round(mape, 2)])

    print(
        "best_alpha:",
        round(best_alpha, 2),
        "best_beta:",
        round(best_beta, 2),
        "best_gamma:",
        round(best_gamma, 2),
        "best_mape:",
        round(best_mape, 4),
    )

    return best_alpha, best_beta, best_gamma, best_mape


def exponential_smoothing_forecast(
    train_df_pandas,
    val_df_pandas,
    seasonal_periods=7,
    trend="add",
    seasonal="mul",
    n_jobs=-1,
):
    """
    Forecasts sales using Exponential Smoothing for each store and item combination using parallel processing.

    Parameters:
    - train_df_pandas: pandas DataFrame, training data in a pandas DataFrame format.
    - val_df_pandas: pandas DataFrame, validation data in a pandas DataFrame format.
    - seasonal_periods: int, number of periods in a complete seasonal cycle (e.g., 7 for weekly seasonality).
    - trend: str, trend component type ('add', 'mul', or None).
    - seasonal: str, seasonal component type ('add', 'mul', or None).
    - n_jobs: int, the number of jobs to run in parallel (-1 means using all processors).

    Returns:
    - forecasts: dict, containing forecasts for each store and item combination.
    """
    # Convert the input Pandas DataFrame to a Polars DataFrame
    train_df = pl.from_pandas(train_df_pandas)
    val_df = pl.from_pandas(val_df_pandas)

    unique_stores = train_df["store_nbr"].unique().to_list()
    unique_items = train_df["item_nbr"].unique().to_list()

    forecasts = {}

    alphas = betas = gammas = np.arange(0.20, 1, 0.10)
    abg = list(itertools.product(alphas, betas, gammas))

    for store in unique_stores:
        store_forecasts = Parallel(n_jobs=n_jobs)(
            delayed(forecast_for_item)(
                store,
                item,
                train_df,
                val_df_pandas,
                abg,
                seasonal_periods,
                trend,
                seasonal,
            )
            for item in unique_items
        )
        forecasts.update(
            {key: forecast for key, forecast in store_forecasts if forecast is not None}
        )

    return forecasts


def forecast_for_item(
    store, item, train_df, val_df_pandas, abg, seasonal_periods, trend, seasonal
):
    # Filter the data for the specific store and item using polars filtering
    train_data = train_df.filter(
        (pl.col("store_nbr") == store) & (pl.col("item_nbr") == item)
    )
    train_data = train_data.sort("date")

    if train_data.height == 0:
        return (store, item), None

    # Convert Polars DataFrame to Pandas (needed for statsmodels)
    train_series = train_data.to_pandas().set_index("date")["unit_sales"]

    # Perform parameter optimization for the store and item
    best_alpha, best_beta, best_gamma, best_mape = hw_model_optimizer(
        train_series,
        abg,
        step=len(val_df_pandas[val_df_pandas["store_nbr"] == store]["date"].unique()),
    )

    # Fit Exponential Smoothing model with optimal parameters
    try:
        final_model = ExponentialSmoothing(
            train_series,
            trend=trend,
            seasonal=seasonal,
            seasonal_periods=seasonal_periods,
        ).fit(
            smoothing_level=best_alpha,
            smoothing_slope=best_beta,
            smoothing_seasonal=best_gamma,
        )

        # Forecast the validation period length
        forecast_length = len(
            val_df_pandas[
                (val_df_pandas["store_nbr"] == store)
                & (val_df_pandas["item_nbr"] == item)
            ]["date"].unique()
        )
        forecast = final_model.forecast(forecast_length)

        return (store, item), forecast

    except Exception as e:
        print(f"Model fitting failed for store {store} item {item}: {e}")
        return (store, item), None


def calculate_metrics(y_true, y_pred):
    """
    Calculates evaluation metrics for the forecast.

    Parameters:
    - y_true: array-like, true values of the target variable.
    - y_pred: array-like, predicted values of the target variable.

    Returns:
    - dict: containing MAPE, Accuracy, and Bias.
    """
    if len(y_true) == 0 or len(y_pred) == 0:
        return {"MAPE": np.nan, "Accuracy": np.nan, "Bias": np.nan}

    mape = mean_absolute_percentage_error(y_true, y_pred)
    accuracy = 1 - mape
    bias = np.mean(y_pred - y_true)
    return {"MAPE": mape, "Accuracy": accuracy, "Bias": bias}


def evaluate_forecasts(forecasts, val_df_pandas):
    """
    Evaluates the forecasts and prints metrics for each store.

    Parameters:
    - forecasts: dict, containing forecasts for each store and item combination.
    - val_df_pandas: pandas DataFrame, validation data containing true values of the target variable.

    Returns:
    - metrics_df: pandas DataFrame, containing metrics for each store, each item, and overall average metrics.
    """
    # Initialize lists to store metrics for each store and item
    all_metrics = []
    store_metrics = {}

    # Iterate over each store and item combination to collect true and predicted values
    for (store, item), forecast in forecasts.items():
        y_true = val_df_pandas[
            (val_df_pandas["store_nbr"] == store) & (val_df_pandas["item_nbr"] == item)
        ]["unit_sales"]
        y_pred = forecast

        if len(y_true) == len(y_pred):
            metrics = calculate_metrics(y_true, y_pred)
            metrics.update({"store_nbr": store, "item_nbr": item})
            all_metrics.append(metrics)

            if store not in store_metrics:
                store_metrics[store] = {"MAPE": [], "Accuracy": [], "Bias": []}
            store_metrics[store]["MAPE"].append(metrics["MAPE"])
            store_metrics[store]["Accuracy"].append(metrics["Accuracy"])
            store_metrics[store]["Bias"].append(metrics["Bias"])

    # Calculate average metrics for each store
    average_store_metrics = []
    for store, metrics in store_metrics.items():
        avg_mape = np.nanmean(metrics["MAPE"])
        avg_accuracy = np.nanmean(metrics["Accuracy"])
        avg_bias = np.nanmean(metrics["Bias"])
        average_store_metrics.append(
            {
                "store_nbr": store,
                "item_nbr": "average",
                "MAPE": avg_mape,
                "Accuracy": avg_accuracy,
                "Bias": avg_bias,
            }
        )

    # Calculate overall average metrics
    overall_mape = np.nanmean(
        [metric["MAPE"] for metric in all_metrics if not np.isnan(metric["MAPE"])]
    )
    overall_accuracy = np.nanmean(
        [
            metric["Accuracy"]
            for metric in all_metrics
            if not np.isnan(metric["Accuracy"])
        ]
    )
    overall_bias = np.nanmean(
        [metric["Bias"] for metric in all_metrics if not np.isnan(metric["Bias"])]
    )
    overall_metrics = {
        "store_nbr": "overall",
        "item_nbr": "overall",
        "MAPE": overall_mape,
        "Accuracy": overall_accuracy,
        "Bias": overall_bias,
    }

    # Convert metrics to DataFrames for better visualization
    metrics_df = pd.DataFrame(all_metrics)
    average_store_metrics_df = pd.DataFrame(average_store_metrics)
    overall_metrics_df = pd.DataFrame([overall_metrics])

    # Print metrics
    print(metrics_df)
    print(average_store_metrics_df)
    print(overall_metrics_df)

    return metrics_df, average_store_metrics_df, overall_metrics_df


# Example usage
if __name__ == "__main__":
    train_df_pandas = pd.read_csv("train_data.csv")
    val_df_pandas = pd.read_csv("val_data.csv")
    forecasts = exponential_smoothing_forecast(train_df_pandas, val_df_pandas)

    # Evaluate forecasts and print metrics
    metrics_df, average_store_metrics_df, overall_metrics_df = evaluate_forecasts(
        forecasts, val_df_pandas
    )

    # The `forecasts` dictionary will contain predictions for each store and item combination
    # The `metrics_df` DataFrame will contain evaluation metrics for each store and item
    # The `average_store_metrics_df` DataFrame will contain average metrics for each store
    # The `overall_metrics_df` DataFrame will contain overall average metrics